# Fine-tune NLLB-200 distilled 600M for Russian -> Abkhazian

Ноутбук готовит LoRA fine-tuning для `facebook/nllb-200-distilled-600M`, мерджит адаптер в модель и сохраняет результат в `./weights`, чтобы `solution.py` мог работать офлайн внутри контейнера.

Ожидаемый минимум данных: параллельный корпус с русской колонкой и абхазской колонкой. Поддерживаются `csv`, `tsv`, `json`, `jsonl`, `parquet`.

## RunPod setup

Рекомендуемый запуск в терминале RunPod:

```bash
uv pip install -r requirements-runpod.txt
jupyter lab --ip 0.0.0.0 --port 8888 --allow-root
```

`requirements-runpod.txt` намеренно не содержит `torch`, потому что в RunPod PyTorch templates CUDA-совместимый PyTorch обычно уже установлен. Если используете чистый образ, ставьте полный `requirements.txt`.

In [ ]:
# Если запускаете ноутбук без предварительной установки зависимостей на RunPod, раскомментируйте:
# %pip install -r requirements-runpod.txt

In [ ]:
from pathlib import Path
import gc
import os

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
import evaluate
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Config

По умолчанию используется найденный в проекте `ab-ru-parallel.csv`. Если путь или названия колонок отличаются, поменяйте `PARALLEL_PATH`, `SRC_COL` и `TGT_COL`.

In [ ]:
MODEL_ID = "facebook/nllb-200-distilled-600M"
SRC_LANG = "rus_Cyrl"
TGT_LANG = "abk_Cyrl"

PARALLEL_PATH = Path("ab-ru-parallel.csv")
SRC_COL = "ru"
TGT_COL = "ab"

OUTPUT_DIR = Path("runs/nllb-ru-ab-lora")
MERGED_DIR = Path("weights")
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
MIN_SRC_CHARS = 2
MIN_TGT_CHARS = 2
MAX_SRC_CHARS = 1000
MAX_TGT_CHARS = 1000
MAX_LENGTH_RATIO = 4.0
VAL_SIZE = 0.05
SEED = 42

USE_4BIT = True
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 3
WARMUP_RATIO = 0.03
SAVE_TOTAL_LIMIT = 2

In [ ]:
def read_parallel(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".tsv":
        df = pd.read_csv(path, sep="\t")
    elif suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".jsonl", ".ndjson"}:
        df = pd.read_json(path, lines=True)
    elif suffix == ".json":
        df = pd.read_json(path)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file extension: {suffix}")
    return df

df = read_parallel(PARALLEL_PATH)
raw_rows = len(df)
print(df.columns.tolist())
df = df[[SRC_COL, TGT_COL]].rename(columns={SRC_COL: "src", TGT_COL: "tgt"})
df = df.dropna()
df["src"] = df["src"].astype(str).str.strip()
df["tgt"] = df["tgt"].astype(str).str.strip()
df = df.drop_duplicates()
df["src_len"] = df["src"].str.len()
df["tgt_len"] = df["tgt"].str.len()
length_ratio = df[["src_len", "tgt_len"]].max(axis=1) / df[["src_len", "tgt_len"]].min(axis=1).clip(lower=1)
df = df[
    df["src_len"].between(MIN_SRC_CHARS, MAX_SRC_CHARS)
    & df["tgt_len"].between(MIN_TGT_CHARS, MAX_TGT_CHARS)
    & (length_ratio <= MAX_LENGTH_RATIO)
]
df = df.drop(columns=["src_len", "tgt_len"]).reset_index(drop=True)
print(f"Rows: raw={raw_rows}, after_cleaning={len(df)}")
df.sample(min(5, len(df)), random_state=SEED)

In [ ]:
dataset = Dataset.from_pandas(df, preserve_index=False)
split = dataset.train_test_split(test_size=VAL_SIZE, seed=SEED)
ds = DatasetDict(train=split["train"], validation=split["test"])
ds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG, tgt_lang=TGT_LANG, use_fast=False)
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert forced_bos_token_id != tokenizer.unk_token_id, f"Missing language token: {TGT_LANG}"

def preprocess(batch):
    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = TGT_LANG
    model_inputs = tokenizer(
        batch["src"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)
tokenized

In [ ]:
bnb_config = None
if USE_4BIT and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.generation_config.forced_bos_token_id = forced_bos_token_id
model.config.forced_bos_token_id = forced_bos_token_id
model.print_trainable_parameters()

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
sacrebleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [" ".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = [[" ".join(label.strip().split())] for label in decoded_labels]
    return {"bleu": sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)["score"]}


In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    bf16=torch.cuda.is_available(),
    fp16=False,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=250,
    save_steps=250,
    save_total_limit=SAVE_TOTAL_LIMIT,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    metric_for_best_model="bleu",
    greater_is_better=True,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate()

## Export merged weights

Этот блок сохраняет полностью слитую модель в `weights/`. Эту папку нужно включить в zip-архив решения.

In [ ]:
trainer.save_model(str(OUTPUT_DIR / "best_adapter"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best_adapter"))

del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Для merge нужен не-квантованный base model. Если не хватает VRAM, выполните этот блок на CPU или отключите USE_4BIT и перезапустите ноутбук.
base = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
from peft import PeftModel
merged = PeftModel.from_pretrained(base, str(OUTPUT_DIR / "best_adapter"))
merged = merged.merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Saved merged model to {MERGED_DIR.resolve()}")

In [ ]:
def translate(texts, batch_size=8):
    tok = AutoTokenizer.from_pretrained(MERGED_DIR, src_lang=SRC_LANG, use_fast=False)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    bos = tok.convert_tokens_to_ids(TGT_LANG)
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH)
        inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
        with torch.inference_mode():
            pred = mdl.generate(**inputs, max_new_tokens=128, num_beams=1, forced_bos_token_id=bos)
        out.extend(tok.batch_decode(pred, skip_special_tokens=True))
    return out

translate(df["src"].head(5).tolist())